In [1]:
!pip install -q trino

In [2]:
from trino.dbapi import connect
from trino.auth import OAuth2Authentication
from redirect_handler import REDIRECT_HANDLER
import urllib3
import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

TRINO_URI = "https://trino-proxy:443"

# Use the Catalog (User 1: Peter)
Now that the catalog is created, we can use it as usual. Lets login again (if asked) as `peter`, this time directly to the catalog so that we don't have to prefix schemas with `lakekeeper`:

In [3]:
conn = connect(
    host=TRINO_URI,
    auth=OAuth2Authentication(REDIRECT_HANDLER),
    http_scheme="https",
    verify=False,
    catalog="lakekeeper" # This line is new
)

In [4]:
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS finance")

Open the following URL in browser for the external authentication:
http://localhost:38191/oauth2/token/initiate/2c4f5d3198aa5519b5b691a52595e693b72860ba36113077c8a31cfcd79d664e


In [5]:
cur.execute("CREATE TABLE IF NOT EXISTS finance.revenue (year INT, revenue DOUBLE)")
cur.execute("INSERT INTO finance.revenue VALUES (2023, 10342.1), (2024, 10645.2)")


In [6]:
# Execute query and fetch all rows
cur.execute("SELECT * FROM finance.revenue")
rows = cur.fetchall()

# Get column names
columns = [desc[0] for desc in cur.description]

# Create DataFrame
df = pd.DataFrame(rows, columns=columns)

# Display DataFrame
print(df)

   year  revenue
0  2023  10342.1
1  2024  10645.2


In [7]:
cur.execute("CREATE TABLE IF NOT EXISTS finance.products (product_id INT, description VARCHAR)")
cur.execute("INSERT INTO finance.products VALUES (1, 'Product 1'), (2, 'Product 2')")


In [8]:
# Execute query and fetch all rows
cur.execute("SELECT * FROM finance.products")
rows = cur.fetchall()

# Get column names
columns = [desc[0] for desc in cur.description]

# Create DataFrame
df = pd.DataFrame(rows, columns=columns)

# Display DataFrame
print(df)

   product_id description
0           1   Product 1
1           2   Product 2
